# VI - Lab 5: Neuronske mreze za regresiju

Rjesenje zadatka: Predvidjanje cijena kuca u Bostonu.


In [ ]:
# (Opcionalno) Ako Colab prijavi da nesto fali, otkomentarisi sljedecu liniju:
# !pip -q install tensorflow scikit-learn matplotlib numpy


In [ ]:
# a) Ucitavanje skupa podataka
import numpy as np
import matplotlib.pyplot as plt

from tensorflow.keras.datasets import boston_housing
from tensorflow.keras import models, layers
from sklearn.preprocessing import MinMaxScaler

np.random.seed(42)

(train_data, train_targets), (test_data, test_targets) = boston_housing.load_data()
print('Podaci ucitani.')


In [ ]:
# b) Oblik podataka i broj atributa
print('train_data shape:', train_data.shape)
print('test_data shape :', test_data.shape)
print('Broj atributa   :', train_data.shape[1])


In [ ]:
# c) Opseg vrijednosti za svaki atribut i target
feature_min = train_data.min(axis=0)
feature_max = train_data.max(axis=0)

print('Opseg vrijednosti po atributima (na trening skupu):')
for i, (mn, mx) in enumerate(zip(feature_min, feature_max), start=1):
    print(f'Atribut {i:2d}: min = {mn:10.4f}, max = {mx:10.4f}')

print('
Target vrijednosti:')
print('train_targets -> min =', np.min(train_targets), ', max =', np.max(train_targets))
print('test_targets  -> min =', np.min(test_targets),  ', max =', np.max(test_targets))


In [ ]:
# d) Skaliranje MinMaxScaler-om (fit SAMO na train)
scaler = MinMaxScaler()
train_data_scaled = scaler.fit_transform(train_data)
test_data_scaled = scaler.transform(test_data)

print('Skaliranje zavrseno.')
print('Train min/max nakon skaliranja:', train_data_scaled.min(), train_data_scaled.max())
print('Test  min/max nakon skaliranja:', test_data_scaled.min(), test_data_scaled.max())


In [ ]:
# e) Pomocna funkcija model_mreze()
def model_mreze():
    model = models.Sequential()
    model.add(layers.Dense(64, activation='relu', input_shape=(train_data_scaled.shape[1],)))
    model.add(layers.Dense(64, activation='relu'))
    model.add(layers.Dense(1))
    return model

m = model_mreze()
m.summary()


In [ ]:
# f) Kompajliranje modela: mse loss, adam optimizer, mae metrika
model_100 = model_mreze()
model_100.compile(optimizer='adam', loss='mse', metrics=['mae'])
print('Model kompajliran.')


In [ ]:
# g) Treniranje 100 epoha, batch_size=1, validation_split=0.1
history_100 = model_100.fit(
    train_data_scaled,
    train_targets,
    epochs=100,
    batch_size=1,
    validation_split=0.1,
    verbose=0
)

final_train_loss_100 = history_100.history['loss'][-1]
final_train_mae_100 = history_100.history['mae'][-1]
final_val_loss_100 = history_100.history['val_loss'][-1]
final_val_mae_100 = history_100.history['val_mae'][-1]

print(f'Kraj treniranja (100 epoha):')
print(f'  Train loss (MSE): {final_train_loss_100:.4f}')
print(f'  Train MAE       : {final_train_mae_100:.4f}')
print(f'  Val loss (MSE)  : {final_val_loss_100:.4f}')
print(f'  Val MAE         : {final_val_mae_100:.4f}')


In [ ]:
# g) Graficki prikaz loss i mae kroz epohe
epochs_100 = range(1, len(history_100.history['loss']) + 1)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(epochs_100, history_100.history['loss'], label='Train MSE')
plt.plot(epochs_100, history_100.history['val_loss'], label='Validation MSE')
plt.xlabel('Epochs')
plt.ylabel('MSE')
plt.title('MSE kroz epohe (100)')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs_100, history_100.history['mae'], label='Train MAE')
plt.plot(epochs_100, history_100.history['val_mae'], label='Validation MAE')
plt.xlabel('Epochs')
plt.ylabel('MAE (hiljade $)')
plt.title('MAE kroz epohe (100)')
plt.legend()

plt.tight_layout()
plt.show()


In [ ]:
# h) Usrednjene vrijednosti validation MAE, bez prvih 10 epoha
def smooth_curve(points, factor=0.9):
    smoothed_points = []
    for point in points:
        if smoothed_points:
            previous = smoothed_points[-1]
            smoothed_points.append(previous * factor + point * (1 - factor))
        else:
            smoothed_points.append(point)
    return smoothed_points

mae_history_100 = history_100.history['val_mae']
smooth_mae_history_100 = smooth_curve(mae_history_100[10:])

plt.figure(figsize=(7, 4))
plt.plot(range(1, len(smooth_mae_history_100) + 1), smooth_mae_history_100)
plt.xlabel('Epochs (od 11. epohe)')
plt.ylabel('Validation MAE (smoothed)')
plt.title('Smoothed Validation MAE - 100 epoha')
plt.show()


In [ ]:
# i) Novi model, treniranje 500 epoha
model_500 = model_mreze()
model_500.compile(optimizer='adam', loss='mse', metrics=['mae'])

history_500 = model_500.fit(
    train_data_scaled,
    train_targets,
    epochs=500,
    batch_size=1,
    validation_split=0.1,
    verbose=0
)

mae_history_500 = history_500.history['val_mae']
smooth_mae_history_500 = smooth_curve(mae_history_500[10:])

plt.figure(figsize=(8, 4))
plt.plot(range(1, len(smooth_mae_history_100) + 1), smooth_mae_history_100, label='100 epoha')
plt.plot(range(1, len(smooth_mae_history_500) + 1), smooth_mae_history_500, label='500 epoha')
plt.xlabel('Epochs (od 11. epohe)')
plt.ylabel('Validation MAE (smoothed)')
plt.title('Poredjenje: 100 vs 500 epoha')
plt.legend()
plt.show()

print('Kratak zakljucak:')
print('- Ako MAE nakon odredjenog broja epoha prestane opadati ili raste, model ulazi u overfitting.')
print('- 500 epoha cesto daje nizu train gresku, ali ne mora dati bolju validation gresku.')


In [ ]:
# j) Evaluacija nad testnim skupom
test_mse_100, test_mae_100 = model_100.evaluate(test_data_scaled, test_targets, verbose=0)
test_mse_500, test_mae_500 = model_500.evaluate(test_data_scaled, test_targets, verbose=0)

print('Test evaluacija (model 100 epoha):')
print(f'  Test MSE: {test_mse_100:.4f}')
print(f'  Test MAE: {test_mae_100:.4f} hiljada $')
print(f'  Prosjecna greska: ${test_mae_100 * 1000:.2f}')

print('
Test evaluacija (model 500 epoha):')
print(f'  Test MSE: {test_mse_500:.4f}')
print(f'  Test MAE: {test_mae_500:.4f} hiljada $')
print(f'  Prosjecna greska: ${test_mae_500 * 1000:.2f}')

if test_mae_100 <= test_mae_500:
    print('
Bolji na testu: model sa 100 epoha (manji MAE).')
else:
    print('
Bolji na testu: model sa 500 epoha (manji MAE).')
